<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/DecoderSingleHead_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch
import torch
from torch import nn
import math
import torch.nn.functional as F

In [ ]:
class Decoder(nn.Module) :
    def __init__(self,d_head,seq_len):
      super().__init__()
      self.d_head = d_head
      self.seq_len = seq_len

      #masked-multihead parameters
      self.Wk = nn.Linear(d_head,d_head)
      self.Wq = nn.Linear(d_head,d_head)
      self.Wv = nn.Linear(d_head,d_head)
      self.norm1 = nn.LayerNorm(d_head)

      self.ffn = nn.Sequential(
          nn.Linear(d_head,d_head*4),
          nn.GELU(),
          nn.Linear(d_head*4,d_head)
      )

      #multihead parameters
      self.Wk_multi = nn.Linear(d_head,d_head)
      self.Wq_multi = nn.Linear(d_head,d_head)
      self.Wv_multi = nn.Linear(d_head,d_head)
      self.norm2 = nn.LayerNorm(d_head)

      #final
      self.norm3 = nn.LayerNorm(d_head)
      self.LinearLayer = nn.Linear(d_head,d_head)

    def maskedSelfAttention(self,input_embedings_ids,padding_mask=None):

      K = self.Wk(input_embedings_ids)
      Q = self.Wq(input_embedings_ids)
      V = self.Wv(input_embedings_ids)

      attention_scores = Q @ K.transpose(-2,-1) #row become column and column become row

      scaled_score = attention_scores / math.sqrt(K.shape[-1])

      if padding_mask is not None:
        scaled_score = scaled_score.masked_fill(
             padding_mask == 0,
            -1e9
          )

      look_ahead_matrix = torch.triu(torch.ones(self.seq_len,self.seq_len),dim=1)

      look_ahead_mask = look_ahead_matrix.masked_fill(
          look_ahead_matrix == 1,
          -1e9
      )

      combined_mask = scaled_score + look_ahead_mask
      attention_weight = F.softmax(combined_mask,dim =-1)
      final_attention = attention_weight @ V

      return final_attention


    def crossMultiheadSelfAttention(self,encoder_output,normalized,padding_mask=None):

      k_multi = self.Wk_multi(encoder_output)
      q_multi = self.Wq_multi(normalized)
      v_multi = self.Wv_multi(encoder_output)

      attention_scores = q_multi @ k_multi.transpose(-2,-1)
      scaled_score = attention_scores / math.sqrt(k_multi.shape[-1])

      if padding_mask is not None:
        scaled_score = scaled_score.masked_fill(
            padding_mask == 0,
            -1e9
        )

      attention_weight = F.softmax(scaled_score,dim =-1)
      final_attention = attention_weight @ v_multi

      return final_attention


    def forward(self,input_embedings_ids,encoder_output,padding_mask=None):

      #masked-multi-head-attention
      attention_output = self.maskedSelfAttention(
          input_embedings_ids,
          padding_mask
        )

      #Add Residual connection
      residual_1 = attention_output + input_embedings_ids

      #normalize
      masked_multihead_output = self.norm1(residual_1)

      #multi-head attention
      cross_multihead_attention = self.crossMultiheadSelfAttention(
          encoder_output,
          masked_multihead_output,
          padding_mask
        )

      #Add Residual connection
      residual_2 = cross_multihead_attention + masked_multihead_output

      #normalize
      normalized2 = self.norm2(residual_2)

      #feed-forward-network
      ffn = self.ffn(normalized2)

      #Add Residual connection
      residual_3 = ffn + normalized2

      #final output
      final_normalized_output = self.norm3(residual_3)

      linear_output = self.LinearLayer(final_normalized_output)
      return linear_output


In [3]:
torch.triu(torch.ones(5,5),1)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

In [8]:
look_ahead_mask = torch.triu(torch.ones(5,5),diagonal=1)
print(look_ahead_mask)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])


In [5]:
look_ahead_mask = look_ahead_mask.masked_fill(
    look_ahead_mask == 1,
    -1e9
)

print(look_ahead_mask)

tensor([[ 0.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, -1.0000e+09, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]])
